# 🧠 Custom Text Sentiment Analyzer

**Goal:** Build an end-to-end NLP pipeline that takes raw unstructured text, cleans it, runs sentiment analysis using `TextBlob` and `NLTK`, and visualizes the emotional distribution using `Matplotlib`.

**Skills demonstrated:**
- Text preprocessing (cleaning, tokenization, stop word removal)
- Sentiment scoring (polarity & subjectivity)
- Data visualization
- Working with unstructured data

---

## 1. Install & Import Libraries

In [ ]:
# Install required packages (run once)
# !pip install textblob nltk matplotlib pandas

import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from textblob import TextBlob

# Download required NLTK resources
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("✅ All libraries loaded successfully!")

## 2. Load the Dataset

We'll use a sample of realistic customer/product reviews as our unstructured text data.
In a real project, this could be loaded from a `.csv`, API, or scraped web data.

In [ ]:
# Sample unstructured text data — simulating real-world review data
raw_reviews = [
    "Absolutely loved this product! It exceeded all my expectations. Highly recommend.",
    "Terrible experience. The item broke after just two days. Waste of money.",
    "It's okay, nothing special. Does what it says but nothing more.",
    "I can't believe how amazing this is. Best purchase I've made all year!",
    "Very disappointed. Poor quality and the customer service was unhelpful.",
    "Pretty decent product for the price. I'd probably buy it again.",
    "Not great, not terrible. It gets the job done but feels cheaply made.",
    "Outstanding quality and fast shipping! Will definitely order again.",
    "Meh. Expected more based on the reviews but it's just average at best.",
    "Complete garbage. DO NOT BUY. Stopped working after a week.",
    "Surprisingly good! Was skeptical at first but I'm really happy with it.",
    "It arrived damaged and the return process was a nightmare.",
    "Works perfectly fine. No complaints from my side, does the job well.",
    "Fantastic! Truly exceeded my expectations. Five stars without hesitation.",
    "Mediocre at best. Wouldn't recommend unless you have no other options."
]

# Load into a DataFrame for structured processing
df = pd.DataFrame({'review': raw_reviews})

print(f"Loaded {len(df)} reviews")
df.head()

## 3. Text Preprocessing Pipeline

Before running sentiment analysis, we need to clean the raw text.
This includes:
- Lowercasing
- Removing punctuation & special characters
- Tokenization (splitting into words)
- Stop word removal (filtering out filler words like 'the', 'is', 'and')

In [ ]:
# Load English stop words from NLTK
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """
    Full text preprocessing pipeline:
    1. Lowercase the text
    2. Remove punctuation and special characters
    3. Tokenize into individual words
    4. Remove stop words
    5. Return cleaned string
    """
    # Step 1: Lowercase
    text = text.lower()
    
    # Step 2: Remove punctuation and non-alphabetic characters
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Step 3: Tokenize
    tokens = word_tokenize(text)
    
    # Step 4: Remove stop words
    filtered_tokens = [word for word in tokens if word not in stop_words]
    
    # Step 5: Rejoin into a clean string
    return ' '.join(filtered_tokens)

# Apply preprocessing to all reviews
df['cleaned_review'] = df['review'].apply(preprocess_text)

# Preview: original vs. cleaned
df[['review', 'cleaned_review']].head(5)

## 4. Sentiment Scoring with TextBlob

TextBlob provides two scores for each piece of text:

| Score | Range | Meaning |
|---|---|---|
| **Polarity** | -1.0 to +1.0 | Negative → Neutral → Positive |
| **Subjectivity** | 0.0 to 1.0 | Objective → Subjective (opinionated) |

We also classify each review as **Positive**, **Neutral**, or **Negative** based on polarity.

In [ ]:
def get_sentiment_scores(text):
    """Run TextBlob sentiment analysis and return polarity & subjectivity."""
    blob = TextBlob(text)
    return blob.sentiment.polarity, blob.sentiment.subjectivity

def classify_sentiment(polarity):
    """Classify polarity score into a human-readable label."""
    if polarity > 0.1:
        return 'Positive'
    elif polarity < -0.1:
        return 'Negative'
    else:
        return 'Neutral'

# Apply scoring to cleaned reviews
df[['polarity', 'subjectivity']] = df['cleaned_review'].apply(
    lambda x: pd.Series(get_sentiment_scores(x))
)

# Classify sentiment label
df['sentiment'] = df['polarity'].apply(classify_sentiment)

# Round for readability
df[['polarity', 'subjectivity']] = df[['polarity', 'subjectivity']].round(3)

print("Sentiment scoring complete!\n")
df[['review', 'polarity', 'subjectivity', 'sentiment']]

## 5. Summary Statistics

In [ ]:
# Overall distribution of sentiment labels
sentiment_counts = df['sentiment'].value_counts()

print("=== Sentiment Distribution ===")
print(sentiment_counts.to_string())
print()

# Descriptive stats for polarity and subjectivity
print("=== Polarity & Subjectivity Stats ===")
print(df[['polarity', 'subjectivity']].describe().round(3).to_string())
print()

# Average polarity per sentiment class
print("=== Average Polarity per Class ===")
print(df.groupby('sentiment')['polarity'].mean().round(3).to_string())

## 6. Data Visualization

We generate three charts to visualize the sentiment insights:
1. **Bar chart** — sentiment label distribution
2. **Histogram** — polarity score distribution across all reviews
3. **Scatter plot** — polarity vs. subjectivity (colored by sentiment)

In [ ]:
# Color palette for consistency
COLOR_MAP = {'Positive': '#2ecc71', 'Neutral': '#f39c12', 'Negative': '#e74c3c'}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Sentiment Analysis — Distribution Overview', fontsize=15, fontweight='bold', y=1.02)

# --- Chart 1: Sentiment Label Distribution (Bar Chart) ---
labels = sentiment_counts.index
colors = [COLOR_MAP[label] for label in labels]
bars = axes[0].bar(labels, sentiment_counts.values, color=colors, edgecolor='white', linewidth=1.2)

# Add count labels on top of each bar
for bar in bars:
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width() / 2, height + 0.1,
                 str(int(height)), ha='center', va='bottom', fontweight='bold')

axes[0].set_title('Sentiment Label Distribution', fontweight='bold')
axes[0].set_xlabel('Sentiment')
axes[0].set_ylabel('Number of Reviews')
axes[0].set_ylim(0, sentiment_counts.max() + 2)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# --- Chart 2: Polarity Score Distribution (Histogram) ---
axes[1].hist(df['polarity'], bins=10, color='#3498db', edgecolor='white', linewidth=0.8)
axes[1].axvline(x=0.1, color='#2ecc71', linestyle='--', linewidth=1.2, label='Positive threshold')
axes[1].axvline(x=-0.1, color='#e74c3c', linestyle='--', linewidth=1.2, label='Negative threshold')
axes[1].set_title('Polarity Score Distribution', fontweight='bold')
axes[1].set_xlabel('Polarity Score (-1 = Negative, +1 = Positive)')
axes[1].set_ylabel('Frequency')
axes[1].legend(fontsize=8)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

# --- Chart 3: Polarity vs. Subjectivity (Scatter Plot) ---
for sentiment, group in df.groupby('sentiment'):
    axes[2].scatter(
        group['polarity'], group['subjectivity'],
        label=sentiment, color=COLOR_MAP[sentiment],
        s=90, alpha=0.85, edgecolors='white', linewidth=0.6
    )

axes[2].axvline(x=0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
axes[2].set_title('Polarity vs. Subjectivity', fontweight='bold')
axes[2].set_xlabel('Polarity')
axes[2].set_ylabel('Subjectivity')
axes[2].legend(title='Sentiment')
axes[2].spines['top'].set_visible(False)
axes[2].spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('sentiment_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n✅ Charts saved as 'sentiment_distribution.png'")

## 7. Custom Input — Analyze Your Own Text

You can test the pipeline on any text by running the cell below.

In [ ]:
def analyze_custom_text(text):
    """
    Takes any raw text input, runs the full pipeline,
    and returns the sentiment result.
    """
    cleaned = preprocess_text(text)
    polarity, subjectivity = get_sentiment_scores(cleaned)
    label = classify_sentiment(polarity)
    
    print(f"Input     : {text}")
    print(f"Cleaned   : {cleaned}")
    print(f"Polarity  : {round(polarity, 3)}")
    print(f"Subjectivity: {round(subjectivity, 3)}")
    print(f"Sentiment : {label}")
    print("-" * 50)

# --- Try your own text here ---
analyze_custom_text("I really enjoyed using this. The interface was intuitive and the results were accurate!")
analyze_custom_text("It's fine I guess, nothing to write home about.")
analyze_custom_text("Absolutely horrible. I regret every penny I spent on this.")

## 8. Key Takeaways & Next Steps

### What this project demonstrates:
- **NLP Pipeline:** End-to-end flow from raw text → cleaned tokens → sentiment scores → visualizations
- **Text Preprocessing:** Lowercasing, regex cleaning, tokenization, stop word removal using NLTK
- **Sentiment Analysis:** Polarity and subjectivity scoring using TextBlob's lexicon-based model
- **Data Visualization:** Automated chart generation with Matplotlib (bar, histogram, scatter)

### Possible extensions:
- Replace sample data with real scraped reviews (Twitter API, Amazon, etc.)
- Compare TextBlob results with a transformer-based model (e.g., `VADER` or `HuggingFace`)
- Build an interactive dashboard using `Streamlit` or `Dash`
- Export results to a `.csv` for stakeholder reporting